# Prompt-injection 16-model study on Kaggle

Start a fresh GPU session and follow [the run guide](kaggle_run.md). Attach the complete repository source and the extracted frozen Phase 3.5 split bundle as separate inputs. Enable Internet for dependency installation, Git cloning, or uncached checkpoints.

Run cells 1–4 to verify the repository and frozen split bundle. Run cells 5–8 with the default TextCNN debug settings, then select DistilBERT in cell 5 and run cells 5–8 again. Debug artifacts go in `/kaggle/working/outputs/debug`. Full runs go in `/kaggle/working/outputs`.

In [ ]:
# 1. Locate and, if needed, stage the complete repository
import hashlib
import importlib.util
import os
import stat
import subprocess
import sys
import tempfile
import zipfile
from pathlib import Path, PurePosixPath

START_DIR = globals().get("START_DIR", Path.cwd())
INPUT_DIR = (START_DIR / Path(os.environ.get("KAGGLE_INPUT_DIR", "/kaggle/input")).expanduser()).resolve()
WORKING_DIR = (START_DIR / Path(os.environ.get("KAGGLE_WORKING_DIR", "/kaggle/working")).expanduser()).resolve()
REPO_SOURCE = os.environ.get("REPO_SOURCE")  # Optional exact directory or ZIP path
REPO_URL = os.environ.get("REPO_URL")  # Optional Git URL when no source is attached

REQUIRED_FILES = (
    "README.md",
    "pyproject.toml",
    "requirements-kaggle.txt",
    "src/__init__.py",
    "src/data/phase35.py",
    "src/kaggle/env.py",
    "src/models/registry.py",
    "src/training/sweep.py",
    "src/eval/tables.py",
    "src/eval/plots.py",
    "configs/kaggle_debug.yaml",
    "configs/experiment.yaml",
    "configs/models/textcnn.yaml",
    "configs/models/distilbert.yaml",
)


def missing_files(root):
    return [name for name in REQUIRED_FILES if not (root / name).is_file()]


def verify_repo(root):
    missing = missing_files(root)
    if missing:
        raise FileNotFoundError(f"Incomplete repository at {root}. Missing: {', '.join(missing)}")
    return root


def repo_candidates(base):
    # Find both complete and partial uploads, including a folder inside the mount.
    markers = ("pyproject.toml", "requirements-kaggle.txt", "src/data/phase35.py")
    return sorted({
        match.parents[len(Path(marker).parts) - 1]
        for marker in markers
        for match in base.rglob(Path(marker).name)
        if match.as_posix().endswith("/" + marker)
    })


def choose_repo(candidates, setting):
    if len(candidates) > 1:
        paths = ", ".join(map(str, candidates))
        raise ValueError(f"Multiple repository sources found: {paths}. Set {setting} to one source.")
    return candidates[0] if candidates else None


def stage_archive(source):
    with source.open("rb") as stream:
        digest = hashlib.file_digest(stream, "sha256").hexdigest()
    destination = WORKING_DIR / f"repo-{digest[:16]}"
    if destination.exists():
        return verify_repo(destination)
    WORKING_DIR.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(dir=WORKING_DIR) as temp:
        unpacked = Path(temp) / "source"
        unpacked.mkdir()
        with zipfile.ZipFile(source) as archive:
            for member in archive.infolist():
                name = PurePosixPath(member.filename)
                if (
                    name.is_absolute()
                    or ".." in name.parts
                    or "\\" in member.filename
                    or ":" in member.filename
                    or stat.S_ISLNK(member.external_attr >> 16)
                ):
                    raise ValueError(f"Unsafe path in repository ZIP: {member.filename}")
            archive.extractall(unpacked)
        candidates = repo_candidates(unpacked)
        root = unpacked if not missing_files(unpacked) else choose_repo(
            [path for path in candidates if not missing_files(path)] or candidates, "REPO_SOURCE"
        )
        verify_repo(root or unpacked).rename(destination)
    return destination


def resolve_repo():
    if REPO_SOURCE:
        source = (START_DIR / Path(REPO_SOURCE).expanduser()).resolve()
        if source.suffix.lower() == ".zip":
            if not source.is_file():
                raise FileNotFoundError(f"Repository ZIP specified by REPO_SOURCE does not exist: {source}")
            return stage_archive(source)
        return verify_repo(source)

    cwd = Path.cwd()
    for root in (cwd, *cwd.parents):
        if not missing_files(root):
            return root

    mounted = repo_candidates(INPUT_DIR)
    attachments = [root for root in mounted if not missing_files(root)]
    attachments += sorted(path for path in INPUT_DIR.rglob("kaggle-repo.zip") if path.is_file())
    source = choose_repo(attachments, "REPO_SOURCE")
    if source is not None:
        return stage_archive(source) if source.is_file() else source

    checkouts = [WORKING_DIR, *sorted(path for path in WORKING_DIR.glob("*") if path.is_dir())]
    root = choose_repo([path for path in checkouts if not missing_files(path)], "REPO_SOURCE")
    if root is not None:
        return root

    if REPO_URL:
        destination = WORKING_DIR / ("repo-git-" + hashlib.sha256(REPO_URL.encode()).hexdigest()[:16])
        if destination.exists():
            return verify_repo(destination)
        WORKING_DIR.mkdir(parents=True, exist_ok=True)
        with tempfile.TemporaryDirectory(dir=WORKING_DIR) as temp:
            checkout = Path(temp) / "repo"
            subprocess.run(["git", "clone", "--depth", "1", "--", REPO_URL, str(checkout)], check=True)
            verify_repo(checkout).rename(destination)
        return destination

    details = "; ".join(
        f"{path} (missing {', '.join(missing_files(path))})"
        for path in mounted + [path for path in checkouts if (path / "src").exists()]
    )
    raise FileNotFoundError(
        f"No complete repository found from {cwd}, {WORKING_DIR}, or {INPUT_DIR}. "
        "Attach the full source or kaggle-repo.zip, set REPO_SOURCE, or provide REPO_URL. "
        f"{details or 'Expected files include pyproject.toml and src/data/phase35.py.'}"
    )


REPO_DIR = resolve_repo().resolve()
os.chdir(REPO_DIR)  # The training CLI resolves per-model YAML relative to the repository.
repo_path = str(REPO_DIR)
sys.path[:] = [repo_path, *(path for path in sys.path if path != repo_path)]

spec = importlib.util.find_spec("src")
if spec is None or spec.origin is None or Path(spec.origin).resolve() != REPO_DIR / "src/__init__.py":
    raise ImportError(f"Active kernel cannot resolve src from {REPO_DIR}. Restart it and run cell 1 first.")
print(f"Repository ready: {REPO_DIR}")

In [ ]:
# 2. Install Kaggle dependencies in the active kernel, retaining Kaggle's PyTorch
from importlib.metadata import version

INSTALL_DEPS = os.environ.get("INSTALL_DEPS", "1") == "1"
if INSTALL_DEPS:
    torch_version = version("torch")
    with tempfile.TemporaryDirectory() as temp:
        constraint = Path(temp) / "torch.txt"
        constraint.write_text(f"torch=={torch_version}\n", encoding="utf-8")
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-r", str(REPO_DIR / "requirements-kaggle.txt"),
             "-c", str(constraint)],
            check=True,
        )
print(f"Kernel Python: {sys.executable}")

In [ ]:
# 3. Inspect the runtime and resolve the extracted Phase 3.5 input
from src.kaggle.env import detect, find_kaggle_splits_dir

SPLITS_SOURCE = os.environ.get("SPLITS_SOURCE")  # Optional exact extracted split directory
if SPLITS_SOURCE:
    SPLITS_SOURCE = START_DIR / Path(SPLITS_SOURCE).expanduser()
SPLITS_DIR = find_kaggle_splits_dir(SPLITS_SOURCE, input_dir=INPUT_DIR)
print(detect(WORKING_DIR).to_dict())
print(f"Frozen split input: {SPLITS_DIR}")

In [ ]:
# 4. Verify the repository import and run the existing frozen-manifest gate
from src.data import phase35
from src.kaggle.env import validate_kaggle_splits
from src.models.registry import list_model_ids

if Path(phase35.__file__).resolve() != REPO_DIR / "src/data/phase35.py":
    raise ImportError(f"phase35 loaded from {phase35.__file__}. Restart the kernel and run from cell 1.")
missing_model_configs = [
    model_id for model_id in list_model_ids()
    if not (REPO_DIR / "configs" / "models" / f"{model_id}.yaml").is_file()
]
if missing_model_configs:
    raise FileNotFoundError(f"Missing per-model YAML files under {REPO_DIR / 'configs/models'}: {missing_model_configs}")
print("REPOSITORY IMPORT VERIFIED")
manifest = validate_kaggle_splits(SPLITS_DIR)
print("LOCAL FROZEN MANIFEST VALIDATED")
print(f"{manifest['version']} | leakage passed: {manifest['leakage_passed']}")
print({split: entry["_total"] for split, entry in manifest["final_counts"].items()})

In [ ]:
# 5. Select a training profile and the jobs for this session
DEBUG = True
MODELS = ["textcnn"]  # Then ["distilbert"]; None uses the selected profile's model list.
SEEDS = [42]  # None uses the profile's default seeds.
BUDGET_MIN = 8 * 60  # Stops launching jobs after this time.
FORCE_BLOCKED = False  # Enable only for a gated model with approved access and HF_TOKEN.
RETRY_FAILED = False  # Enable after correcting the error in an earlier attempt.

CONFIG = REPO_DIR / "configs" / ("kaggle_debug.yaml" if DEBUG else "experiment.yaml")
OUTPUT_DIR = WORKING_DIR / "outputs"
if DEBUG:
    OUTPUT_DIR /= "debug"
RUNS_DIR = OUTPUT_DIR / "runs"
SWEEP_MANIFEST = OUTPUT_DIR / "sweep" / "manifest.json"
print(f"Config: {CONFIG}\nOutputs: {OUTPUT_DIR}")

In [ ]:
# 6. Run the resumable sweep
from src.training import sweep

argv = [
    "--config", str(CONFIG), "--splits-dir", str(SPLITS_DIR), "--runs-dir", str(RUNS_DIR),
    "--manifest", str(SWEEP_MANIFEST), "--budget-minutes", str(BUDGET_MIN),
]
if MODELS:
    argv += ["--models", *MODELS]
if SEEDS:
    argv += ["--seeds", *map(str, SEEDS)]
if FORCE_BLOCKED:
    argv.append("--force-blocked")
if RETRY_FAILED:
    argv.append("--retry-failed")
if sweep.main(argv):
    raise RuntimeError(f"Training jobs failed. Review {SWEEP_MANIFEST} for the job errors.")

In [ ]:
# 7. Export tables and figures from saved predictions
from src.eval import plots, tables

tables.main(["--runs", str(RUNS_DIR), "--out", str(OUTPUT_DIR / "tables"), "--n-boot", "500"])
plot_args = [
    "--runs", str(RUNS_DIR), "--out", str(OUTPUT_DIR / "figures"), "--splits-dir", str(SPLITS_DIR),
]
if DEBUG:
    plot_args += ["--tag", "DEBUG"]
plots.main(plot_args)

In [ ]:
# 8. Export the registry with status derived from this profile's artifacts
from src.models import registry

registry.main([
    "--runs-dir", str(RUNS_DIR), "--smoke-dir", str(OUTPUT_DIR / "smoke"),
    "--export", str(OUTPUT_DIR / "registry.json"), "--markdown", "",
])